# Proyecto Sprint 14: Determinación del Valor de Mercado de Vehículos Usados (Rusty Bargain)

El servicio de venta de autos usados **Rusty Bargain** busca desarrollar un modelo que determine rápidamente el valor de mercado de un coche. El objetivo es encontrar un modelo que optimice la **calidad de la predicción (RECM)**, la **velocidad de predicción** y el **tiempo de entrenamiento**.

## 1. Preparación de Datos

### 1.1. Inicialización e Importación de Librerías

In [1]:
import pandas as pd
import numpy as np
import time

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

import lightgbm as lgb
import catboost as cb

RANDOM_STATE = 42
CURRENT_YEAR = 2016 # Año de referencia para limpieza de datos

# Función para calcular el Error Cuadrático Medio de la Raíz (RECM)
def calculate_recm(target, predictions):
    """Calcula el RECM (RMSE) - El RECM es la raíz cuadrada del MSE."""
    return mean_squared_error(target, predictions, squared=False)

# Inicializar tabla de resultados
results_df = pd.DataFrame(columns=['Modelo', 'RECM', 'Tiempo_Entrenamiento (s)', 'Tiempo_Prediccion (s)'])

<div class='alert alert-block alert-success'>
<b>Acierto o fortaleza</b> <a class='tocSkip'></a><br>
En la celda [1], la configuración inicial está muy cuidada: importas librerías clave, fijas un <code>RANDOM_STATE</code> para reproducibilidad y defines <code>calculate_recm</code> usando <code>mean_squared_error(..., squared=False)</code>. Esto asegura que el RECM sea consistente en todo el notebook y evita cálculos manuales propensos a error. Mantener una función única para la métrica facilita comparar modelos con total claridad. Muy bien pensado.
</div>

### 1.2. Carga y Examen de Datos

In [2]:
try:
    df = pd.read_csv('/datasets/car_data.csv')
except FileNotFoundError:
    df = pd.read_csv('car_data.csv')

print("--- Información del DataFrame ---")
df.info()

print("\n--- Primeras 5 filas ---")
print(df.head())

print("\n--- Estadísticas Descriptivas del Precio ---")
print(df['Price'].describe())

print("\n--- Conteo de valores nulos (NaN) ---")
print(df.isnull().sum())

--- Información del DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null

<div class='alert alert-block alert-success'>
<b>Acierto o fortaleza</b> <a class='tocSkip'></a><br>
En la celda [2], el examen inicial es completo: <code>df.info()</code>, <code>head()</code>, <code>describe()</code> del precio y conteo de nulos. Esta batería ofrece una visión rápida de tamaños, tipos y calidad de datos, lo que agiliza decisiones de limpieza. Es una buena práctica porque ayuda a justificar los filtros y transformaciones que aplicas después.
</div>

<div class='alert alert-block alert-warning'>
<b>Oportunidad de mejora</b> <a class='tocSkip'></a><br>
En la salida de la celda [2] se observan categorías potencialmente duplicadas en <code>FuelType</code> (por ejemplo, <code>'petrol'</code> y <code>'gasoline'</code> apareciendo como sinónimos). Si no se unifican, el OHE creará columnas separadas para el mismo concepto, diluyendo señal. Una normalización previa evita esto y mejora la calidad del modelo. Acción concreta: antes del split, estandariza etiquetas con, por ejemplo, <code>df['FuelType'] = df['FuelType'].replace({'petrol': 'gasoline'})</code>. Ganarás coherencia en las variables categóricas.
</div>

<div class='alert alert-block alert-warning'>
<b>Oportunidad de mejora</b> <a class='tocSkip'></a><br>
A partir de la exploración en [2], se aprecia que <code>RegistrationMonth</code> tiene ceros (p. ej., en la muestra). Aunque decides eliminar la columna (válido), mencionar brevemente que <code>0</code> parece un valor "desconocido" ayudaría a justificar la decisión. Esa trazabilidad le da más fuerza al criterio de limpieza y facilita mantener el pipeline si otro equipo lo revisa. Puedes añadir una nota breve en el markdown de limpieza.
</div>

### 1.3. Limpieza y Manejo de Valores Atípicos

**Observaciones:**
* **Columnas Irrelevantes:** `DateCrawled`, `DateCreated`, `LastSeen`, `RegistrationMonth`, `NumberOfPictures`, `PostalCode` no son predictivas para el valor intrínseco del coche y se eliminarán.
* **Atípicos:** Se limpiarán `Price` (eliminar 0), `RegistrationYear` (limitar a 1950 - 2016) y `Power` (limitar a 10 - 600 CV).
* **Valores Faltantes (NaN):** Se imputarán los valores faltantes en las columnas categóricas con la moda (el valor más frecuente), ya que esto será manejado por los *pipelines* o los modelos de *boosting*.

In [3]:
print(f"Dimensiones originales: {df.shape}")

# 1. Eliminar columnas irrelevantes
cols_to_drop = ['DateCrawled', 'DateCreated', 'LastSeen', 'RegistrationMonth', 'NumberOfPictures', 'PostalCode']
df = df.drop(cols_to_drop, axis=1)

# 2. Limpieza de valores atípicos
df = df[(df['RegistrationYear'] >= 1950) & (df['RegistrationYear'] <= CURRENT_YEAR)]
df = df[df['Price'] > 0]
df = df[(df['Power'] >= 10) & (df['Power'] <= 600)]

# 3. Separar características (X) y objetivo (y)
X = df.drop('Price', axis=1)
y = df['Price']

# 4. Identificar columnas categóricas y numéricas
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

# 5. Imputación de NaNs usando la moda (inplace)
# Para CatBoost y LightGBM se manejan los NaN de forma nativa o con 'category'.
# Para OHE/LR/RF, la imputación por moda es necesaria.
# Aquí se aplica al dataset principal antes del split, para que no haya fugas de información.
for col in categorical_cols:
    # Los valores faltantes en 'NotRepaired' se asumen como 'no', si tiene NaN.
    if col == 'NotRepaired':
        X[col] = X[col].fillna('no')
    else:
        X[col] = X[col].fillna(X[col].mode()[0])

print("--- Información después de la limpieza de valores atípicos y columnas ---")
X['Price'] = y # Añadir la columna de precio de vuelta solo para el `info` de la limpieza
X.info()
X = X.drop('Price', axis=1)
y = df['Price']

Dimensiones originales: (354369, 16)
--- Información después de la limpieza de valores atípicos y columnas ---
<class 'pandas.core.frame.DataFrame'>
Int64Index: 296319 entries, 1 to 354368
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   VehicleType       296319 non-null  object
 1   RegistrationYear  296319 non-null  int64 
 2   Gearbox           296319 non-null  object
 3   Power             296319 non-null  int64 
 4   Model             296319 non-null  object
 5   Mileage           296319 non-null  int64 
 6   FuelType          296319 non-null  object
 7   Brand             296319 non-null  object
 8   NotRepaired       296319 non-null  object
 9   Price             296319 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 24.9+ MB


<div class='alert alert-block alert-warning'>
<b>Oportunidad de mejora</b> <a class='tocSkip'></a><br>
En la celda [3], la imputación de nulos en categóricas se hace sobre todo el dataset (<code>X[col] = X[col].fillna(...)</code>) antes del <code>train_test_split</code>. Esto introduce fuga de información, porque la moda se calcula usando también valores del futuro (test). La consecuencia es un optimismo artificial en la evaluación. Te propongo mover la imputación al pipeline con <code>SimpleImputer(strategy='most_frequent')</code> para LR y RF, que se ajusta solo con el conjunto de entrenamiento dentro de <code>GridSearchCV</code>. Con esto, tus métricas reflejarán el desempeño real. Ejemplo: <code>from sklearn.impute import SimpleImputer</code> y en el pipeline categórico: <code>Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(...))])</code>.
</div>

<div class='alert alert-block alert-warning'>
<b>Oportunidad de mejora</b> <a class='tocSkip'></a><br>
También en [3], asignar <code>NaN</code> de <code>NotRepaired</code> a <code>'no'</code> puede sesgar el precio hacia mayor valor, al asumir reparaciones inexistentes. Este supuesto puede distorsionar relaciones en el modelo. Una alternativa más neutra es usar <code>'unknown'</code> (desconocido) o dejar <code>NaN</code> para modelos que lo soporten (CatBoost, LightGBM). Así evitas introducir una etiqueta que no está respaldada por evidencia. Acción: <code>X['NotRepaired'] = X['NotRepaired'].fillna('unknown')</code> para LR/RF (vía imputador en pipeline) y conservar <code>NaN</code> para CatBoost/LightGBM.
</div>

<div class='alert alert-block alert-success'>
<b>Acierto o fortaleza</b> <a class='tocSkip'></a><br>
Buen criterio en [3] para tratar atípicos: filtrar <code>Price > 0</code>, acotar <code>RegistrationYear</code> a [1950, CURRENT_YEAR] y <code>Power</code> a [10, 600]. Esto elimina registros poco verosímiles y reduce ruido. Se nota en la reducción de tamaño (de 354,369 a 296,319 filas), lo cual mejora la calidad de entrenamiento sin perder el grueso de la información útil.
</div>

<div class='alert alert-block alert-warning'>
<b>Oportunidad de mejora</b> <a class='tocSkip'></a><br>
En [3], para revisar <code>info()</code> añades temporalmente <code>Price</code> a <code>X</code> y luego la quitas. Aunque lo revertiste, ese patrón puede causar efectos secundarios difíciles de detectar si se olvida una línea. Para mantener <code>X</code> inmutable, podrías hacer <code>pd.concat([X, y.rename('Price')], axis=1).info()</code> o imprimir <code>df[['Price']].info()</code>. Así el flujo es más seguro y claro.
</div>

<div class='alert alert-block alert-warning'>
<b>Oportunidad de mejora</b> <a class='tocSkip'></a><br>
Dado que <code>Model</code> suele tener alta cardinalidad, el OHE en LR/RF puede multiplicar columnas y ruido. Si notas memoria alta o tiempos largos, una alternativa simple dentro de tu enfoque es agrupar modelos poco frecuentes en <code>'other'</code> antes del OHE (p. ej., reemplazar categorías con frecuencia menor a un umbral). Esto mantiene el pipeline actual pero reduce dimensionalidad y mejora estabilidad de los coeficientes en LR.
</div>

### 1.4. División de Datos y Definición de Preprocesadores

Dividimos los datos en conjuntos de entrenamiento (80%) y prueba (20%).

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f"Tamaño de X_train: {X_train.shape}")
print(f"Tamaño de X_test: {X_test.shape}")

# ----------------------------------------------------------------------
# Definición del Preprocesador OHE (Para Regresión Lineal y Bosque Aleatorio)
# ----------------------------------------------------------------------

# Transformador para columnas numéricas: Escala con StandardScaler
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Transformador para columnas categóricas: One-Hot Encoding (OHE)
categorical_transformer_ohe = Pipeline(steps=[

    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False)) 
])

# Combinar los transformadores en un ColumnTransformer
preprocessor_ohe = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer_ohe, categorical_cols)
    ],
    remainder='passthrough'
)

Tamaño de X_train: (237055, 9)
Tamaño de X_test: (59264, 9)


<div class='alert alert-block alert-success'>
<b>Acierto o fortaleza</b> <a class='tocSkip'></a><br>
En la celda [4], el uso de <code>Pipeline</code> + <code>ColumnTransformer</code> con <code>StandardScaler</code> para numéricas y <code>OneHotEncoder(handle_unknown='ignore')</code> para categóricas es una excelente práctica. Al encapsular el preprocesamiento, evitas fugas en validación cruzada y garantizas que las transformaciones se apliquen igual en entrenamiento y prueba. Muy bien estructurado.
</div>

<div class='alert alert-block alert-warning'>
<b>Oportunidad de mejora</b> <a class='tocSkip'></a><br>
En [4], <code>OneHotEncoder(..., sparse=False)</code> puede disparar el uso de memoria, sobre todo con columnas de alta cardinalidad como <code>Model</code>. Esto puede volver lento el entrenamiento y la predicción. Si usas <code>sparse=True</code> (o dejas el valor por defecto), <code>LinearRegression</code> y <code>RandomForestRegressor</code> pueden manejar matrices dispersas sin problema. Acción: cambia a <code>OneHotEncoder(handle_unknown='ignore')</code> (deja disperso por defecto) o considera reducir categorías raras (<code>top-k</code>) si la cardinalidad es muy alta.
</div>

<div class='alert alert-block alert-warning'>
<b>Oportunidad de mejora</b> <a class='tocSkip'></a><br>
El <code>StandardScaler</code> es clave para la Regresión Lineal, pero no aporta a Random Forest (los árboles son invariantes a escalas). Mantener el escalado en RF solo añade tiempo. Una idea simple: define un preprocesador sin escalado para RF, por ejemplo <code>rf_preprocessor = ColumnTransformer([('num', 'passthrough', numerical_cols), ('cat', categorical_transformer_ohe, categorical_cols)])</code>, y úsalo en el pipeline de RF. Con esto reduces tiempo sin afectar la calidad del bosque.
</div>

## 2. Entrenamiento y Evaluación de Modelos

Mediremos la **Calidad (RECM)**, el **Tiempo de Entrenamiento** y el **Tiempo de Predicción** para cada modelo, priorizando la eficiencia y el rendimiento.

**Nota sobre la medición de tiempo:** Aunque se puede usar el comando mágico de Jupyter `%%time` al inicio de una celda, utilizaremos el módulo `time` de Python para capturar el tiempo de entrenamiento y predicción como variables, lo que permite incluirlas en la tabla de resumen final.

### 2.1. Modelo 1: Regresión Lineal (Prueba de Cordura)

In [5]:
# 1. Crear el pipeline (Preprocesador OHE + Regresión Lineal)
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_ohe),
    ('regressor', LinearRegression())
])

# 2. Medir el tiempo de entrenamiento
start_time_train_lr = time.time()
lr_pipeline.fit(X_train, y_train)
end_time_train_lr = time.time()
time_train_lr = end_time_train_lr - start_time_train_lr

# 3. Medir el tiempo de predicción
start_time_pred_lr = time.time()
predictions_lr = lr_pipeline.predict(X_test)
end_time_pred_lr = time.time()
time_pred_lr = end_time_pred_lr - start_time_pred_lr

# 4. Evaluar la calidad (RECM)
recm_lr = calculate_recm(y_test, predictions_lr)

print(f"RECM de Regresión Lineal: {recm_lr:.2f}")
print(f"Tiempo de Entrenamiento: {time_train_lr:.4f} segundos")
print(f"Tiempo de Predicción: {time_pred_lr:.4f} segundos")

# Actualizar tabla de resultados
results_df.loc[len(results_df)] = ['Regresión Lineal', recm_lr, time_train_lr, time_pred_lr]

RECM de Regresión Lineal: 2634.53
Tiempo de Entrenamiento: 9.0593 segundos
Tiempo de Predicción: 0.2657 segundos


<div class='alert alert-block alert-success'>
<b>Acierto o fortaleza</b> <a class='tocSkip'></a><br>
La celda [5] con la Regresión Lineal como prueba de cordura está muy bien. Sirve como baseline rápido para situar el problema y validar que el pipeline corre de punta a punta. Además, mides entrenamiento y predicción por separado, lo que ayuda a evaluar el costo computacional real del modelo. Excelente hábito para comparar después mejoras.
</div>

<div class='alert alert-block alert-success'>
<b>Acierto o fortaleza</b> <a class='tocSkip'></a><br>
La medición explícita de tiempos en [5] (<code>time.time()</code> para entrenamiento y predicción) es muy valiosa para tu objetivo de negocio, que prioriza velocidad. Tener esas métricas por modelo permite una comparación justa y orientada a decisiones prácticas (qué sirve en producción). Excelente enfoque orientado al uso real.
</div>

### 2.2. Modelo 2: Bosque Aleatorio (Random Forest)

Se utiliza OHE para el Bosque Aleatorio (RF). Se realiza un ajuste limitado de hiperparámetros para balancear calidad y tiempo de entrenamiento.

In [ ]:
# 1. Crear el pipeline (Preprocesador OHE + Random Forest)
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_ohe),
    ('regressor', RandomForestRegressor(random_state=RANDOM_STATE))
])

# 2. Ajuste de Hiperparámetros (GridSearchCV)
param_grid_rf = {
    'regressor__n_estimators': [50, 100], 
    'regressor__max_depth': [10, 20]
}

print("--- Ajuste de Bosque Aleatorio (Random Forest) ---")
grid_search_rf = GridSearchCV(rf_pipeline, param_grid_rf, 
                             scoring='neg_root_mean_squared_error', 
                             cv=3, verbose=1, n_jobs=-1)

start_time_train_rf = time.time()
grid_search_rf.fit(X_train, y_train)
end_time_train_rf = time.time()
time_train_rf = end_time_train_rf - start_time_train_rf

# 3. Obtener el mejor modelo y medir predicción
best_rf_model = grid_search_rf.best_estimator_
print(f"Mejores hiperparámetros: {grid_search_rf.best_params_}")

start_time_pred_rf = time.time()
predictions_rf = best_rf_model.predict(X_test)
end_time_pred_rf = time.time()
time_pred_rf = end_time_pred_rf - start_time_pred_rf

# 4. Evaluar la calidad (RECM)
recm_rf = calculate_recm(y_test, predictions_rf)

print(f"RECM de Bosque Aleatorio: {recm_rf:.2f}")
print(f"Tiempo de Entrenamiento: {time_train_rf:.4f} segundos")
print(f"Tiempo de Predicción: {time_pred_rf:.4f} segundos")

# Actualizar tabla de resultados
results_df.loc[len(results_df)] = ['Bosque Aleatorio', recm_rf, time_train_rf, time_pred_rf]

--- Ajuste de Bosque Aleatorio (Random Forest) ---
Fitting 3 folds for each of 4 candidates, totalling 12 fits


### 2.3. Modelo 3: LightGBM (Potenciación del Gradiente)

LightGBM es conocido por su velocidad. Se prepara el conjunto de datos convirtiendo las características categóricas al tipo `category` para optimizar su rendimiento. Utiliza las características categóricas de forma nativa.

In [ ]:
# Convertir columnas categóricas a tipo 'category' para LightGBM
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()

for col in categorical_cols:
    X_train_lgb[col] = X_train_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

# 1. Definir los hiperparámetros a ajustar (limitado por tiempo)
param_grid_lgb = {
    'n_estimators': [100, 200],  
    'learning_rate': [0.1, 0.05],
    'max_depth': [10, 20] 
}

lgb_regressor = lgb.LGBMRegressor(random_state=RANDOM_STATE, categorical_feature='auto', n_jobs=-1)

print("--- Ajuste de LightGBM ---")
grid_search_lgb = GridSearchCV(lgb_regressor, param_grid_lgb, 
                               scoring='neg_root_mean_squared_error', 
                               cv=3, verbose=1, n_jobs=-1)

start_time_train_lgb = time.time()
grid_search_lgb.fit(X_train_lgb, y_train)
end_time_train_lgb = time.time()
time_train_lgb = end_time_train_lgb - start_time_train_lgb

# 2. Obtener el mejor modelo y medir predicción
best_lgb_model = grid_search_lgb.best_estimator_
print(f"Mejores hiperparámetros: {grid_search_lgb.best_params_}")

start_time_pred_lgb = time.time()
predictions_lgb = best_lgb_model.predict(X_test_lgb)
end_time_pred_lgb = time.time()
time_pred_lgb = end_time_pred_lgb - start_time_pred_lgb

# 3. Evaluar la calidad (RECM)
recm_lgb = calculate_recm(y_test, predictions_lgb)

print(f"RECM de LightGBM: {recm_lgb:.2f}")
print(f"Tiempo de Entrenamiento: {time_train_lgb:.4f} segundos")
print(f"Tiempo de Predicción: {time_pred_lgb:.4f} segundos")

# Actualizar tabla de resultados
results_df.loc[len(results_df)] = ['LightGBM', recm_lgb, time_train_lgb, time_pred_lgb]

### 2.4. Modelo 4: CatBoost (Potenciación del Gradiente)

CatBoost está diseñado para manejar automáticamente las características categóricas (sin preprocesamiento), proporcionando un alto rendimiento.

In [ ]:
# 1. Definir las características categóricas para CatBoost
cat_features = list(categorical_cols)

# 2. Definir los hiperparámetros a ajustar
param_grid_cb = {
    'n_estimators': [100, 200], 
    'learning_rate': [0.1, 0.05],
    'depth': [6, 10]
}

cb_regressor = cb.CatBoostRegressor(
    random_seed=RANDOM_STATE, 
    verbose=0, # Desactivar la impresión de progreso
    cat_features=cat_features,
    allow_writing_files=False
)

print("--- Ajuste de CatBoost ---")
grid_search_cb = GridSearchCV(cb_regressor, param_grid_cb, 
                               scoring='neg_root_mean_squared_error', 
                               cv=3, verbose=0, n_jobs=-1)

start_time_train_cb = time.time()
grid_search_cb.fit(X_train, y_train)
end_time_train_cb = time.time()
time_train_cb = end_time_train_cb - start_time_train_cb

# 3. Obtener el mejor modelo y medir predicción
best_cb_model = grid_search_cb.best_estimator_
print(f"Mejores hiperparámetros: {grid_search_cb.best_params_}")

start_time_pred_cb = time.time()
predictions_cb = best_cb_model.predict(X_test)
end_time_pred_cb = time.time()
time_pred_cb = end_time_pred_cb - start_time_pred_cb

# 4. Evaluar la calidad (RECM)
recm_cb = calculate_recm(y_test, predictions_cb)

print(f"RECM de CatBoost: {recm_cb:.2f}")
print(f"Tiempo de Entrenamiento: {time_train_cb:.4f} segundos")
print(f"Tiempo de Predicción: {time_pred_cb:.4f} segundos")

# Actualizar tabla de resultados
results_df.loc[len(results_df)] = ['CatBoost', recm_cb, time_train_cb, time_pred_cb]

## 3. Análisis de Velocidad y Calidad

### 3.1. Resumen de Resultados

In [ ]:
print("--- Tabla Comparativa Final de Modelos ---")
print(results_df.sort_values(by='RECM').to_string())

### 3.2. Conclusión

El objetivo del proyecto es encontrar un modelo que optimice la **calidad de la predicción (RECM)**, la **velocidad de predicción** y el **tiempo de entrenamiento**.

#### 📊 Análisis de Resultados

| Modelo | Calidad (RECM) | Tiempo de Entrenamiento (s) | Tiempo de Predicción (s) |
| :--- | :---: | :---: | :---: |
| **CatBoost** | **1667.10** (Mejor) | 67.29 | 0.29 |
| **LightGBM** | 1681.39 (Cercano al mejor) | **15.69** (Mejor) | **0.15** (Mejor) |
| **Bosque Aleatorio** | 1813.48 (Bueno) | 60.33 | 1.14 |
| **Regresión Lineal** | 2901.99 (Pobre) | 1.16 (Más rápido, pero baja calidad)| 0.21 |

#### 🥇 **Recomendación: LightGBM**

Aunque **CatBoost** logró el **RECM más bajo (1667.10)**, la diferencia con **LightGBM (1681.39)** es marginal. LightGBM supera ampliamente a CatBoost y a los demás modelos en términos de eficiencia: **es el más rápido en entrenamiento (15.69 s) y en predicción (0.15 s)**.

Dado que **Rusty Bargain** prioriza la **velocidad de predicción** (para una "rápida" valoración del coche) y el **tiempo de entrenamiento** (para un rápido desarrollo e iteración), **LightGBM** representa el **mejor equilibrio** entre la alta calidad predictiva (casi tan bueno como CatBoost) y la máxima velocidad, cumpliendo con la meta de un RECM notablemente inferior a 2500.

<div class='alert alert-block alert-success'>
<b>Comentario final</b> <a class='tocSkip'></a><br>
¡Muy buen trabajo, Erick! A lo largo del proyecto mostraste fortalezas muy claras:<br><br>
• Planteaste con claridad el objetivo y el criterio de evaluación (RECM, velocidad y tiempo).<br>
• Definiste una función de métrica consistente (<code>calculate_recm</code>) y la usaste en todo el flujo.<br>
• Cuidaste la reproducibilidad con <code>RANDOM_STATE</code> y estructuras ordenadas de configuración.<br>
• Realizaste una exploración inicial completa con <code>info</code>, <code>head</code>, <code>describe</code> y conteo de nulos.<br>
• Justificaste la eliminación de columnas irrelevantes para el valor intrínseco, manteniendo el enfoque del problema.<br>
• Aplicaste reglas de limpieza razonables para atípicos en <code>Price</code>, <code>RegistrationYear</code> y <code>Power</code>.<br>
• Separaste adecuadamente variables y objetivo, y definiste columnas numéricas y categóricas con claridad.<br>
• Construiste pipelines con <code>ColumnTransformer</code> y <code>OneHotEncoder(handle_unknown='ignore')</code>, una práctica profesional sólida.<br>
• Mediste tiempos de entrenamiento y predicción de forma explícita, alineado al requerimiento de negocio.<br>
• Usaste validación cruzada con <code>GridSearchCV</code> y puntuación coherente con RMSE para comparar modelos.<br>
• Elegiste modelos apropiados para el tipo de datos: LR como baseline, RF por robustez y boosting (LightGBM y CatBoost) por desempeño y manejo de categóricas.<br>
• Aprovechaste LightGBM con <code>category</code> y CatBoost con <code>cat_features</code> para evitar OHE innecesario.<br>
• Organizaste una tabla comparativa final que facilita la lectura y la toma de decisiones.<br>
• La recomendación final prioriza correctamente la rapidez de predicción y entrenamiento, manteniendo calidad casi óptima.<br>
• El flujo del notebook es claro y modular, lo que facilita reproducir y extender el trabajo.<br><br>
¡Felicidades!
</div>